# TDC ADMET Classification with HMH Cached Features

This notebook is separate from the Python 3.14 RLHF notebook. Use the kernel **Python 3.11 (TDC)**.

It loads TDC ADMET classification datasets, inspects their SMILES/labels, converts molecules with RDKit, computes cached HMH spectral-diffusion graph features, and trains an efficient HMH MLP classifier.

In [ ]:
# 0) Environment check
import sys
print(sys.executable)
print(sys.version)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from tqdm.auto import tqdm
from IPython.display import display

try:
    import tdc
    from tdc.single_pred import ADME, Tox
    print("TDC import OK")
except Exception as exc:
    raise RuntimeError("This notebook must use the Python 3.11 (TDC) kernel with pytdc installed.") from exc

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# 1) TDC ADMET dataset catalog
TDC_TASKS = {
    "AMES": ("Tox", "AMES"),
    "hERG": ("Tox", "hERG"),
    "hERG_Karim": ("Tox", "hERG_Karim"),
    "DILI": ("Tox", "DILI"),
    "HIA": ("ADME", "HIA_Hou"),
    "BBB": ("ADME", "BBB_Martins"),
    "CYP2C9_inhibition": ("ADME", "CYP2C9_Veith"),
    "CYP2D6_inhibition": ("ADME", "CYP2D6_Veith"),
    "CYP3A4_inhibition": ("ADME", "CYP3A4_Veith"),
}

def load_tdc_task(task_key, data_dir="data/TDC"):
    loader_name, dataset_name = TDC_TASKS[task_key]
    loader_cls = Tox if loader_name == "Tox" else ADME
    dataset = loader_cls(name=dataset_name, path=data_dir)
    df = dataset.get_data().copy()
    df = df.dropna(subset=["Drug", "Y"]).reset_index(drop=True)
    df["Y"] = df["Y"].astype(float)
    return dataset_name, df

summary = []
for key in TDC_TASKS:
    try:
        name, df_tmp = load_tdc_task(key)
        counts = df_tmp["Y"].value_counts().sort_index().to_dict()
        summary.append({"task_key": key, "tdc_name": name, "n": len(df_tmp), "label_counts": counts, "positive_rate": float((df_tmp["Y"] == 1).mean())})
    except Exception as exc:
        summary.append({"task_key": key, "tdc_name": TDC_TASKS[key][1], "n": np.nan, "label_counts": repr(exc), "positive_rate": np.nan})

tdc_summary = pd.DataFrame(summary)
display(tdc_summary)


In [ ]:
# 2) Choose and inspect one dataset
# Good first small runs: hERG, DILI, HIA, BBB. Larger: AMES, hERG_Karim, CYP*.
TASK_KEY = "AMES"
MAX_ROWS = None  # set e.g. 1000 for a quick smoke run

tdc_name, df = load_tdc_task(TASK_KEY)
if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print("Task:", TASK_KEY, "| TDC name:", tdc_name, "| shape:", df.shape)
display(df.head(10))
display(df["Y"].value_counts().sort_index().to_frame("count"))
display(df["Y"].value_counts(normalize=True).sort_index().to_frame("fraction"))
print("Example SMILES:")
for s in df["Drug"].head(5):
    print(s)


In [ ]:
# 3) RDKit molecule featurization
ATOM_TYPES = ["C", "N", "O", "S", "F", "P", "Cl", "Br", "I", "B", "Si", "Se", "other"]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]

def one_hot(value, choices):
    out = [0.0] * len(choices)
    try:
        idx = choices.index(value)
    except ValueError:
        idx = len(choices) - 1
    out[idx] = 1.0
    return out

def atom_features(atom):
    symbol = atom.GetSymbol()
    symbol = symbol if symbol in ATOM_TYPES[:-1] else "other"
    return np.array(
        one_hot(symbol, ATOM_TYPES) + [
            float(atom.GetDegree()),
            float(atom.GetFormalCharge()),
            float(atom.GetTotalNumHs()),
            float(atom.GetIsAromatic()),
            float(atom.GetMass()) / 100.0,
        ],
        dtype=np.float64,
    )

def smiles_to_arrays(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = np.vstack([atom_features(atom) for atom in mol.GetAtoms()])
    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append((i, j))
        edges.append((j, i))
    return x, edges

# Quick check
example = smiles_to_arrays(df.loc[0, "Drug"])
print("Example node feature shape:", example[0].shape, "num directed edges:", len(example[1]))


In [ ]:
# 4) HMH spectral-diffusion feature builder
def normalized_laplacian_from_adjacency(A):
    degree = np.asarray(A.sum(axis=1)).ravel()
    degree_safe = np.maximum(degree, 1e-12)
    D_inv = sp.diags(1.0 / np.sqrt(degree_safe))
    L = sp.eye(A.shape[0], format="csr") - D_inv @ A @ D_inv
    return L, degree

def bottom_eigenvectors(L_aug, k_latent):
    n = L_aug.shape[0]
    try:
        _, U = spla.eigsh(L_aug, k=k_latent, which="SM", maxiter=max(1000, 20 * n), tol=1e-6)
    except Exception:
        dense = L_aug.toarray()
        dense = 0.5 * (dense + dense.T)
        vals, vecs = np.linalg.eigh(dense)
        U = vecs[:, np.argsort(vals)[:k_latent]]
    return U

def hmh_vector_from_smiles(smiles, k_feat=8, k_diff=8, t=0.6, lam=0.08, alpha=0.5, r_probe=12, k_spec=16):
    parsed = smiles_to_arrays(smiles)
    if parsed is None:
        return None
    x_np, edges = parsed
    n = x_np.shape[0]
    g_feat = x_np.mean(axis=0)

    if n <= 2 or len(edges) == 0:
        return np.concatenate([g_feat, np.zeros(k_spec, dtype=np.float64)]).astype(np.float32)

    rows = np.array([e[0] for e in edges], dtype=np.int64)
    cols = np.array([e[1] for e in edges], dtype=np.int64)
    A = sp.coo_matrix((np.ones(len(edges), dtype=np.float64), (rows, cols)), shape=(n, n)).tocsr()
    A = A.maximum(A.T)
    A.setdiag(0)
    A.eliminate_zeros()
    if A.nnz == 0:
        return np.concatenate([g_feat, np.zeros(k_spec, dtype=np.float64)]).astype(np.float32)

    L_top, degree = normalized_laplacian_from_adjacency(A)

    # Feature-kNN graph over atoms in the molecule.
    kf = min(max(1, int(k_feat)), max(1, n - 1))
    diff = x_np[:, None, :] - x_np[None, :, :]
    dist2 = np.sum(diff * diff, axis=-1)
    np.fill_diagonal(dist2, np.inf)
    idx = np.argpartition(dist2, kth=kf - 1, axis=1)[:, :kf]
    f_rows = np.repeat(np.arange(n), kf)
    f_cols = idx.reshape(-1)
    d2 = dist2[f_rows, f_cols]
    pos = d2[np.isfinite(d2) & (d2 > 0)]
    sigma = float(np.sqrt(np.median(pos))) if pos.size else 1.0
    sigma = max(sigma, 1e-12)
    weights = np.exp(-d2 / (2.0 * sigma * sigma))
    W_feat = sp.csr_matrix((weights, (f_rows, f_cols)), shape=(n, n))
    W_feat = W_feat.maximum(W_feat.T)
    W_feat.setdiag(0)
    W_feat.eliminate_zeros()
    L_feat, _ = normalized_laplacian_from_adjacency(W_feat)

    L_mix = alpha * L_top + (1.0 - alpha) * L_feat

    rng = np.random.default_rng(SEED)
    omega = rng.standard_normal((n, min(r_probe, n)))
    Z = spla.expm_multiply((-t) * L_mix, omega)
    Z = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)

    kd = min(max(1, int(k_diff)), max(1, n - 1))
    zdiff = Z[:, None, :] - Z[None, :, :]
    zdist2 = np.sum(zdiff * zdiff, axis=-1)
    np.fill_diagonal(zdist2, np.inf)
    zidx = np.argpartition(zdist2, kth=kd - 1, axis=1)[:, :kd]

    in_ball = np.zeros((n, n), dtype=bool)
    np.fill_diagonal(in_ball, True)
    for i in range(n):
        in_ball[i, zidx[i]] = True
    complement = ~in_ball
    np.fill_diagonal(complement, False)

    inv_degree = 1.0 / np.maximum(degree, 1e-12)
    M = sp.csr_matrix(np.where(complement, inv_degree[:, None] * inv_degree[None, :], 0.0))
    M = M.maximum(M.T)
    M.setdiag(0)
    M.eliminate_zeros()

    L_aug = (L_mix + lam * M).tocsr()
    k_latent = min(k_spec, max(2, n - 1))
    U = bottom_eigenvectors(L_aug, k_latent)
    g_spec_raw = U.mean(axis=0)
    g_spec = np.zeros(k_spec, dtype=np.float64)
    g_spec[: min(k_spec, g_spec_raw.shape[0])] = g_spec_raw[: min(k_spec, g_spec_raw.shape[0])]
    return np.concatenate([g_feat, g_spec]).astype(np.float32)

print("HMH feature dim example:", hmh_vector_from_smiles(df.loc[0, "Drug"]).shape)


In [ ]:
# 5) Cached HMH feature matrix
from pathlib import Path
from joblib import Parallel, delayed

CACHE_DIR = Path("data/hmh_tdc_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

HMH_PARAMS = dict(k_feat=8, k_diff=8, t=0.6, lam=0.08, alpha=0.5, r_probe=12, k_spec=16)
N_JOBS = 4  # increase if you want faster preprocessing and your Mac has spare cores

def cache_path_for(task_key, max_rows, params):
    param_tag = "_".join([f"{k}-{v}" for k, v in params.items()])
    row_tag = "all" if max_rows is None else str(max_rows)
    return CACHE_DIR / f"{task_key}_{row_tag}_{param_tag}.pt"

def compute_or_load_hmh_features(df, task_key=TASK_KEY, max_rows=MAX_ROWS, params=HMH_PARAMS, force_recompute=False):
    path = cache_path_for(task_key, max_rows, params)
    if path.exists() and not force_recompute:
        print("Loading cached HMH features:", path)
        obj = torch.load(path, map_location="cpu", weights_only=False)
        return obj["X"], obj["y"], obj["smiles"], obj["valid_index"]

    print("Computing HMH features. This is the expensive step, but it is cached after this run.")
    smiles_list = df["Drug"].astype(str).tolist()
    labels = df["Y"].astype(float).to_numpy()

    vecs = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(hmh_vector_from_smiles)(s, **params) for s in tqdm(smiles_list, desc=f"HMH {task_key}")
    )

    valid = [i for i, v in enumerate(vecs) if v is not None and np.isfinite(v).all()]
    if not valid:
        raise RuntimeError("No valid molecules after RDKit/HMH featurization.")
    X = torch.tensor(np.vstack([vecs[i] for i in valid]), dtype=torch.float32)
    y = torch.tensor(labels[valid], dtype=torch.float32).view(-1, 1)
    smiles_valid = [smiles_list[i] for i in valid]

    obj = {"X": X, "y": y, "smiles": smiles_valid, "valid_index": valid, "params": params, "task_key": task_key}
    torch.save(obj, path)
    print("Saved cache:", path)
    print("Valid molecules:", len(valid), "/", len(df))
    return X, y, smiles_valid, valid

X, y, smiles_valid, valid_index = compute_or_load_hmh_features(df)
print("X:", X.shape, "y:", y.shape)


In [ ]:
# 6) Train/validation/test split
indices = np.arange(len(y))
y_np = y.view(-1).numpy()
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=y_np)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED, stratify=y_np[temp_idx])

mu = X[train_idx].mean(dim=0, keepdim=True)
sd = X[train_idx].std(dim=0, keepdim=True)
sd[sd < 1e-8] = 1.0
Xn = (X - mu) / sd

print("train/val/test:", len(train_idx), len(val_idx), len(test_idx))
print("train positive rate:", float(y[train_idx].mean()))
print("val positive rate:", float(y[val_idx].mean()))
print("test positive rate:", float(y[test_idx].mean()))


In [ ]:
# 7) Efficient HMH MLP classifier
class HMHGraphClassifier(nn.Module):
    def __init__(self, in_dim, hidden=128, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

def metrics_from_logits(y_true, logits):
    yt = y_true.detach().cpu().view(-1).numpy()
    prob = torch.sigmoid(logits.detach().cpu()).view(-1).numpy()
    pred = (prob >= 0.5).astype(int)
    return {
        "roc_auc": float(roc_auc_score(yt, prob)) if len(np.unique(yt)) == 2 else np.nan,
        "avg_precision": float(average_precision_score(yt, prob)),
        "accuracy@0.5": float(accuracy_score(yt, pred)),
    }

def make_loader(idx, batch_size=128, shuffle=False):
    ds = torch.utils.data.TensorDataset(Xn[idx], y[idx])
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_idx, batch_size=128, shuffle=True)
val_loader = make_loader(val_idx, batch_size=256, shuffle=False)
test_loader = make_loader(test_idx, batch_size=256, shuffle=False)

model = HMHGraphClassifier(in_dim=Xn.size(1), hidden=128, dropout=0.25).to(DEVICE)
num_pos = float(y[train_idx].sum())
num_neg = float(len(train_idx) - num_pos)
pos_weight = torch.tensor([num_neg / max(num_pos, 1.0)], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def run_epoch(loader, train=False):
    model.train(train)
    ys, logits_all = [], []
    total_loss, total_n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * yb.size(0)
        total_n += yb.size(0)
        ys.append(yb.detach().cpu())
        logits_all.append(logits.detach().cpu())
    m = metrics_from_logits(torch.cat(ys), torch.cat(logits_all))
    m["loss"] = total_loss / max(total_n, 1)
    return m


In [ ]:
# 8) Train and evaluate
EPOCHS = 120
PATIENCE = 200

best_val = -float("inf")
best_state = None
best_test = None
wait = 0

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    te = run_epoch(test_loader, train=False)

    if va["roc_auc"] > best_val:
        best_val = va["roc_auc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_test = te.copy()
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train AUC {tr['roc_auc']:.4f} | val AUC {va['roc_auc']:.4f} | test AUC {te['roc_auc']:.4f}")

    if wait >= PATIENCE:
        print("Early stopping at epoch", epoch)
        break

if best_state is not None:
    model.load_state_dict(best_state)
print("Best-by-val test metrics:", best_test)


In [ ]:
# 9) Prediction preview
model.eval()
with torch.no_grad():
    logits = model(Xn[test_idx].to(DEVICE)).cpu()
    prob = torch.sigmoid(logits).view(-1).numpy()

preview = pd.DataFrame({
    "smiles": [smiles_valid[i] for i in test_idx[:20]],
    "true": y[test_idx].view(-1).numpy()[:20],
    "prob_positive": prob[:20],
})
display(preview)


In [ ]:
# 10) Optional loop over several TDC ADMET tasks using cached HMH features
def run_task_quick(task_key, max_rows=None, epochs=80):
    global TASK_KEY, MAX_ROWS, df, X, y, smiles_valid, valid_index, Xn, train_idx, val_idx, test_idx
    TASK_KEY = task_key
    MAX_ROWS = max_rows
    _, df = load_tdc_task(TASK_KEY)
    if max_rows is not None and len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=SEED).reset_index(drop=True)
    X, y, smiles_valid, valid_index = compute_or_load_hmh_features(df, task_key=TASK_KEY, max_rows=MAX_ROWS)
    idx = np.arange(len(y))
    yn = y.view(-1).numpy()
    train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=yn)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED, stratify=yn[temp_idx])
    mu = X[train_idx].mean(0, keepdim=True)
    sd = X[train_idx].std(0, keepdim=True)
    sd[sd < 1e-8] = 1.0
    Xn = (X - mu) / sd

    tr_loader = make_loader(train_idx, batch_size=128, shuffle=True)
    va_loader = make_loader(val_idx, batch_size=256, shuffle=False)
    te_loader = make_loader(test_idx, batch_size=256, shuffle=False)
    local_model = HMHGraphClassifier(Xn.size(1), hidden=128, dropout=0.25).to(DEVICE)
    pos = float(y[train_idx].sum())
    neg = float(len(train_idx) - pos)
    local_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / max(pos, 1.0)], device=DEVICE))
    local_opt = torch.optim.AdamW(local_model.parameters(), lr=1e-3, weight_decay=1e-4)

    def step(loader, train=False):
        local_model.train(train)
        ys, logs = [], []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = local_model(xb)
            loss = local_criterion(out, yb)
            if train:
                local_opt.zero_grad(); loss.backward(); local_opt.step()
            ys.append(yb.cpu()); logs.append(out.detach().cpu())
        return metrics_from_logits(torch.cat(ys), torch.cat(logs))

    best = -float("inf")
    best_test = None
    for ep in range(1, epochs + 1):
        step(tr_loader, train=True)
        vm = step(va_loader, train=False)
        tm = step(te_loader, train=False)
        if vm["roc_auc"] > best:
            best = vm["roc_auc"]
            best_test = tm
    return {"task": task_key, "n": len(y), **best_test}

# Uncomment for a cached benchmark sweep. First run computes features; reruns load cache.
# tasks_to_run = ["AMES", "hERG", "DILI", "HIA", "BBB"]
# results = [run_task_quick(t, max_rows=None, epochs=80) for t in tasks_to_run]
# display(pd.DataFrame(results))
